In [125]:
"""
One-time script to preprocess raw UCSC XENA clinical data and save as a new csv
"""

'\nOne-time script to preprocess raw UCSC XENA clinical data and save as a new csv\n'

## Load Data

In [126]:
# Dataset is GDC TCGA BRCA via UCSC Xena
# Phenotype
# https://xenabrowser.net/datapages/?dataset=TCGA-BRCA.clinical.tsv&host=https%3A%2F%2Fgdc.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

In [127]:
import os
import pandas as pd
import numpy as np
import seaborn as sns

In [128]:
df = pd.read_csv("../data/raw/TCGA-BRCA.clinical.tsv", 
                  delimiter='\t', index_col=0)
df.head()

,id,disease_type,case_id,submitter_id,primary_site,alcohol_history.exposures,race.demographic,gender.demographic,ethnicity.demographic,vital_status.demographic,...,days_to_collection.samples,initial_weight.samples,preservation_method.samples,pathology_report_uuid.samples,oct_embedded.samples,specimen_type.samples,days_to_sample_procurement.samples,is_ffpe.samples,tissue_type.samples,annotations.samples
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-BH-A0W3-01A,3c612e12-6de8-44fa-a095-805c45474821,Ductal and Lobular Neoplasms,3c612e12-6de8-44fa-a095-805c45474821,TCGA-BH-A0W3,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,85.0,120.0,OCT,801A4E2F-E26E-424F-BF42-CD0D9CD62BCE,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-AR-A24V-01A,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,Ductal and Lobular Neoplasms,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,TCGA-AR-A24V,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,1720.0,400.0,OCT,468CD293-C9F7-43C6-A40A-18FCDD22F6AA,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-01A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,280.0,OCT,CF6E29A2-FAE6-45BB-B625-33877887A89E,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-11A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,830.0,OCT,NaN,True,Solid Tissue,NaN,False,Normal,NaN
TCGA-AC-A8OQ-01A,dfaabd03-2d40-4422-b210-caf112ff4229,Ductal and Lobular Neoplasms,dfaabd03-2d40-4422-b210-caf112ff4229,TCGA-AC-A8OQ,Breast,Not Reported,black or african american,female,not hispanic or latino,Alive,...,742.0,100.0,Unknown,FFA6F9F3-71C1-4AF9-B9F7-0466550EBC90,False,Solid Tissue,NaN,False,Tumor,NaN


In [129]:
df.shape

(1255, 84)

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1255 entries, TCGA-BH-A0W3-01A to TCGA-A7-A0D9-11A
Data columns (total 84 columns):
 #   Column                                                     Non-Null Count  Dtype  
---  ------                                                     --------------  -----  
 0   id                                                         1255 non-null   object 
 1   disease_type                                               1255 non-null   object 
 2   case_id                                                    1255 non-null   object 
 3   submitter_id                                               1255 non-null   object 
 4   primary_site                                               1255 non-null   object 
 5   alcohol_history.exposures                                  1254 non-null   object 
 6   race.demographic                                           1254 non-null   object 
 7   gender.demographic                                         1254 non-null  

In [131]:
# STEP 1 - Only use primary tumor samples
# primary tumor is denoted as TCGA-xx-xxxx-01

rows_to_drop = [row for row in df.index if row[13:15] != '01']
df.drop(index=rows_to_drop, inplace=True)
df.head(15)
print(f"Dropped {len(rows_to_drop)} rows that were not primary tumors. {df.shape[0]} rows remaining.")

Dropped 144 rows that were not primary tumors. 1111 rows remaining.


In [132]:
# save outcome features for later Cox PH testing of covariates

os_events = df['vital_status.demographic'].apply(lambda x: 0 if x == "Alive" else 1)
os_times = df['days_to_last_follow_up.diagnoses'].fillna(0) + df['days_to_death.demographic'].fillna(0)

print(os_events.loc['TCGA-E9-A5FK-01A'])
print(os_times.loc['TCGA-E9-A5FK-01A'])
print(os_events.loc['TCGA-BH-A203-01A'])
print(os_times.loc['TCGA-BH-A203-01A'])

0
812.0
1
1174.0


## Filter Clinical Variables

### Remove Unusable Columns

In [133]:
# STEP 2 - drop features that are >50% null

n_notnull = {}
for col in df.columns:
    n_notnull[col] = int(df[col].notnull().sum())

cols_to_drop = [col for col in n_notnull.keys()
                if n_notnull[col] < df.shape[0]*0.5]

print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns that had >50% nulls. {df.shape[1]} columns remaining.")

['year_of_death.demographic', 'days_to_death.demographic', 'entity_submitter_id.annotations', 'notes.annotations', 'submitter_id.annotations', 'classification.annotations', 'entity_id.annotations', 'created_datetime.annotations', 'annotation_id.annotations', 'entity_type.annotations', 'updated_datetime.annotations', 'case_id.annotations', 'state.annotations', 'category.annotations', 'status.annotations', 'case_submitter_id.annotations', 'days_to_sample_procurement.samples', 'annotations.samples']
Dropped 18 columns that had >50% nulls. 66 columns remaining.


In [134]:
# STEP 3 - drop features that are >50% "Not Reported"

n_notreported = {}
for col in df.columns:
    n_notreported[col] = len([x for x in df[col] 
                              if x == "Not Reported" or x == "not reported"])

cols_to_drop = [col for col in n_notreported.keys()
                if n_notreported[col] > df.shape[0]*0.5]

print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns that had >50% Not Reported. {df.shape[1]} columns remaining.")

['alcohol_history.exposures', 'last_known_disease_status.diagnoses', 'classification_of_tumor.diagnoses', 'tumor_grade.diagnoses', 'progression_or_recurrence.diagnoses', 'composition.samples']
Dropped 6 columns that had >50% Not Reported. 60 columns remaining.


In [135]:
# STEP 4 - drop features with only a single value

uniform_cols = []
for col in df.columns:
    n_unique = len(df[col].unique())
    if n_unique == 1:
        uniform_cols.append(col)

print(uniform_cols)

df.drop(columns=uniform_cols, inplace=True)
print(f"Dropped {len(uniform_cols)} columns that had only one unique value. {df.shape[1]} columns remaining.")

['primary_site', 'primary_site.project', 'project_id.project', 'disease_type.project', 'name.project', 'name.program.project', 'project.tissue_source_site', 'bcr_id.tissue_source_site', 'sample_type_id.samples', 'tumor_descriptor.samples', 'sample_type.samples', 'specimen_type.samples', 'tissue_type.samples']
Dropped 13 columns that had only one unique value. 47 columns remaining.


In [136]:
df.columns

Index(['id', 'disease_type', 'case_id', 'submitter_id', 'race.demographic',
       'gender.demographic', 'ethnicity.demographic',
       'vital_status.demographic', 'age_at_index.demographic',
       'days_to_birth.demographic', 'year_of_birth.demographic',
       'tissue_source_site_id.tissue_source_site', 'code.tissue_source_site',
       'name.tissue_source_site', 'synchronous_malignancy.diagnoses',
       'ajcc_pathologic_stage.diagnoses', 'days_to_diagnosis.diagnoses',
       'tissue_or_organ_of_origin.diagnoses',
       'days_to_last_follow_up.diagnoses', 'age_at_diagnosis.diagnoses',
       'primary_diagnosis.diagnoses', 'prior_malignancy.diagnoses',
       'year_of_diagnosis.diagnoses', 'prior_treatment.diagnoses',
       'ajcc_staging_system_edition.diagnoses', 'ajcc_pathologic_t.diagnoses',
       'morphology.diagnoses', 'ajcc_pathologic_n.diagnoses',
       'ajcc_pathologic_m.diagnoses', 'icd_10_code.diagnoses',
       'site_of_resection_or_biopsy.diagnoses',
       'age_at_

### Remove Problematic / Useless Columns

In [137]:
# STEP 5 - drop features that are administrative / IDs

admin_cols = ['id', 'case_id', 'submitter_id',
              'tissue_source_site_id.tissue_source_site',
              'code.tissue_source_site',
              'name.tissue_source_site',
              'ajcc_staging_system_edition.diagnoses',
              'treatment_id.treatments.diagnoses',
              'submitter_id.treatments.diagnoses',
              'created_datetime.treatments.diagnoses',
              'updated_datetime.treatments.diagnoses',
              'sample_id.samples',
              'pathology_report_uuid.samples',
              'is_ffpe.samples',
              'preservation_method.samples',
              'oct_embedded.samples',
              'initial_weight.samples']

df.drop(columns=admin_cols, inplace=True)
print(f"Dropped {len(admin_cols)} columns that correspond to IDs or administrative labels. {df.shape[1]} columns remaining.")

Dropped 17 columns that correspond to IDs or administrative labels. 30 columns remaining.


In [138]:
# STEP 6 - drop spurious features or features that leak survival outcome
# dropping treatment since that risks shortcut learning; treatment represents a clinician's prediction - ignore!
# dropping year_of_diagnosis since that's not clinical, but represents temporal differences

outcome_cols = ['vital_status.demographic',
                'days_to_last_follow_up.diagnoses',
                'days_to_collection.samples',
                'treatment_type.treatments.diagnoses',
                'treatment_or_therapy.treatments.diagnoses',
                'prior_treatment.diagnoses',
                'year_of_diagnosis.diagnoses']

df.drop(columns=outcome_cols, inplace=True)
print(f"Dropped {len(outcome_cols)} columns that leak the survival outcome. {df.shape[1]} columns remaining.")

Dropped 7 columns that leak the survival outcome. 23 columns remaining.


In [139]:
# STEP 7 - drop features with low / negligible variance

low_variance_cols = ['days_to_diagnosis.diagnoses',
                     'disease_type',
                     'gender.demographic',
                     'tissue_or_organ_of_origin.diagnoses',
                     'state.treatments.diagnoses',
                     'site_of_resection_or_biopsy.diagnoses']

for col in low_variance_cols:
    print(df[col].value_counts())

# all are confirmed same value, redundant values, or highly skewed

df.drop(columns=low_variance_cols, inplace=True)
print(f"Dropped {len(low_variance_cols)} columns that had little-to-no variance. {df.shape[1]} columns remaining.")

days_to_diagnosis.diagnoses
0.0    1110
Name: count, dtype: int64
disease_type
Ductal and Lobular Neoplasms             1065
Complex Epithelial Neoplasms               16
Cystic, Mucinous and Serous Neoplasms      16
Epithelial Neoplasms, NOS                   5
Adenomas and Adenocarcinomas                3
Fibroepithelial Neoplasms                   2
Squamous Cell Neoplasms                     2
Basal Cell Neoplasms                        1
Adnexal and Skin Appendage Neoplasms        1
Name: count, dtype: int64
gender.demographic
female    1098
male        12
Name: count, dtype: int64
tissue_or_organ_of_origin.diagnoses
Breast, NOS                       1100
Lower-inner quadrant of breast       3
Overlapping lesion of breast         2
Upper-outer quadrant of breast       2
Upper-inner quadrant of breast       2
Lower-outer quadrant of breast       1
Name: count, dtype: int64
state.treatments.diagnoses
['released', 'released']    1110
Name: count, dtype: int64
site_of_resection_or_bio

In [140]:
# STEP 8 - collapse collinear / redunant features (age and staging)

# multiple columns for age, just use one
age_kept = 'age_at_index.demographic'
# choose individual T/N/M stage features over the composite AJCC stage
stage_kept = ['ajcc_pathologic_t', 'ajcc_pathologic_n', 'ajcc_pathologic_m']

redundant_cols = ['days_to_birth.demographic',
                  'year_of_birth.demographic',
                  'age_at_diagnosis.diagnoses',
                  'age_at_earliest_diagnosis.diagnoses.xena_derived',
                  'age_at_earliest_diagnosis_in_years.diagnoses.xena_derived',
                  'ajcc_pathologic_stage.diagnoses',
                  'icd_10_code.diagnoses',
                  'morphology.diagnoses']

df.drop(columns=redundant_cols, inplace=True)
print(f"Dropped {len(redundant_cols)} columns that were collinear/redundant. {df.shape[1]} columns remaining.")

Dropped 8 columns that were collinear/redundant. 9 columns remaining.


In [141]:
df.columns

Index(['race.demographic', 'ethnicity.demographic', 'age_at_index.demographic',
       'synchronous_malignancy.diagnoses', 'primary_diagnosis.diagnoses',
       'prior_malignancy.diagnoses', 'ajcc_pathologic_t.diagnoses',
       'ajcc_pathologic_n.diagnoses', 'ajcc_pathologic_m.diagnoses'],
      dtype='object')

### Statistical Investigation

In [142]:
from lifelines import CoxPHFitter

In [143]:
# STEP 9 - fit Cox survival model to each remaining variable

cox_results = pd.DataFrame(columns=['covariate', 'log_likelihood', 'hazard_ratio',
                                    'confidence_interval', '-log2(p)'])

for col in df.columns:
    covariate_df = pd.DataFrame({
        'time': os_times,
        'event': os_events,
        col: pd.factorize(df[col])[0]
    })
    
    cph = CoxPHFitter()
    cph.fit(covariate_df, duration_col='time', event_col='event')
    
    ll = round(cph.log_likelihood_, 2)
    hr = round(cph.hazard_ratios_.loc[col], 3)
    ci = (round(cph.summary.loc[col, 'exp(coef) lower 95%'], 3),
          round(cph.summary.loc[col, 'exp(coef) upper 95%'], 3))
    log_p = round(cph.summary.loc[col, '-log2(p)'], 3)
    
    cox_results.loc[len(cox_results)] = [col, ll, hr, ci, log_p]

cox_results

,covariate,log_likelihood,hazard_ratio,confidence_interval,-log2(p)
0,race.demographic,-853.47,1.027,"(0.828, 1.273)",0.308
1,ethnicity.demographic,-848.06,0.464,"(0.269, 0.8)",7.453
2,age_at_index.demographic,-853.46,1.001,"(0.992, 1.011)",0.368
3,synchronous_malignancy.diagnoses,-853.24,1.317,"(0.642, 2.702)",1.146
4,primary_diagnosis.diagnoses,-852.70,1.028,"(0.987, 1.071)",2.459
5,prior_malignancy.diagnoses,-853.25,1.306,"(0.64, 2.665)",1.111
6,ajcc_pathologic_t.diagnoses,-849.68,1.120,"(1.04, 1.206)",8.469
7,ajcc_pathologic_n.diagnoses,-849.08,1.065,"(1.022, 1.11)",8.501
8,ajcc_pathologic_m.diagnoses,-849.88,1.456,"(1.13, 1.876)",8.095


In [144]:
# STEP 10 - examine distribution of variables

if not os.path.exists("../data/processed/clinical_distributions/"):
    os.mkdir("../data/processed/clinical_distributions/")


for col in df.columns:
    covariate_df = pd.DataFrame({
        'time': os_times,
        'event': os_events,
        col: pd.factorize(df[col])[0]
    })
    
    # histogram if continuous
    if len(df[col].unique()) > 10:
        axes = sns.histplot(covariate_df, x=col, hue='event', kde=True)
        fig = axes.get_figure()
        fig.savefig(f"../data/processed/clinical_distributions/{col}_hist.png", dpi=100)
        fig.clear()
    
    # categorical bar chart if discrete
    else:
        axes = sns.countplot(covariate_df, x=col, hue='event')
        fig = axes.get_figure()
        fig.savefig(f"../data/processed/clinical_distributions/{col}_bar.png", dpi=100)
        fig.clear()

<Figure size 640x480 with 0 Axes>

In [145]:
# Drop features with -log2(p) < 4

low_significance_cols = cox_results[cox_results['-log2(p)'] < 4]['covariate']
print(low_significance_cols.values)

# we are okay dropping age here since small dataset and T/N/M features + genes will help capture age discrepancies

df.drop(columns=low_significance_cols, inplace=True)
print(f"Dropped {len(low_significance_cols)} columns that had low statistical significance w.r.t. survival. {df.shape[1]} columns remaining.")

['race.demographic' 'age_at_index.demographic'
 'synchronous_malignancy.diagnoses' 'primary_diagnosis.diagnoses'
 'prior_malignancy.diagnoses']
Dropped 5 columns that had low statistical significance w.r.t. survival. 4 columns remaining.


In [146]:
# Drop ethnicity, despite statistical and clinical significance, since poor distribution ( <50 hispanic/latino)
df['ethnicity.demographic'].value_counts()

df.drop(columns=['ethnicity.demographic'], inplace=True)

In [147]:
# Final clinical variables - just going with T/N/M staging!

rename_dict = {
    'ajcc_pathologic_t.diagnoses': 't_stage',
    'ajcc_pathologic_n.diagnoses': 'n_stage',
    'ajcc_pathologic_m.diagnoses': 'm_stage'
}

df.rename(columns=rename_dict, inplace=True)

print(df.shape)
df.head()

(1111, 3)


,t_stage,n_stage,m_stage
sample,,,
TCGA-BH-A0W3-01A,T1c,N1a,M0
TCGA-AR-A24V-01A,T2,N1,M0
TCGA-E9-A1NE-01A,T2,N1,M0
TCGA-AC-A8OQ-01A,T2,N1a,MX
TCGA-AC-A23C-01A,T2,N1,MX


In [148]:
for col in df.columns:
    print(df[col].value_counts())

t_stage
T2     641
T1c    225
T3     139
T1      42
T4b     28
T1b     16
T4       9
T4d      3
TX       3
T2a      1
T3a      1
T2b      1
T1a      1
Name: count, dtype: int64
n_stage
N0           338
N1a          168
N0 (i-)      158
N1           128
N2a           64
N2            56
N3a           47
N1mi          38
N1b           32
N0 (i+)       28
N3            26
NX            20
N3b            3
N1c            2
N0 (mol+)      1
N3c            1
Name: count, dtype: int64
m_stage
M0          913
MX          169
M1           22
cM0 (i+)      6
Name: count, dtype: int64


## Encode & Save Dataset

In [149]:
# STEP 11 - drop any remaining null rows

df.dropna(inplace=True)
print(df.shape)

(1110, 3)


In [150]:
# STEP 12 - ordinal encode stage features

# T - tumor stage
t_encode = {
    'TX': np.nan,
    'T1': 1, 'T1a': 1, 'T1b': 2, 'T1c': 3,
    'T2': 4, 'T2a': 4, 'T2b': 5,
    'T3': 7, 'T3a': 7,      # severity jump
    'T4': 10, 'T4b': 11, 'T4d': 12     # severity jump
}
df['t_stage'] = df['t_stage'].replace(t_encode)


# N - lymph node
n_encode = {
    'NX': np.nan,
    'N0': 0, 'N0 (i-)': 0, 'N0 (i+)': 0, 'N0 (mol+)': 0,
    'N1mi': 1, 'N1': 2, 'N1a': 2, 'N1b': 2, 'N1c': 3,
    'N2': 5, 'N2a': 5,
    'N3': 8, 'N3a': 8, 'N3b': 8, 'N3c': 9   # severity jump
}
df['n_stage'] = df['n_stage'].replace(n_encode)


# M - metastasis
m_encode = {
    'MX': 0,    # usually M0
    'M0': 0,
    'cM0 (i+)': 0,
    'M1': 1
}
df['m_stage'] = df['m_stage'].replace(m_encode)

# median encode the unable-to-asses (X) codes
for stage in ['t_stage', 'n_stage']:
    median = np.median(df[stage].dropna())
    print(f"Filled {df[stage].isna().sum()} nulls in {stage} with median {median}")
    df[stage] = df[stage].fillna(median)

clinical = df.copy()

Filled 3 nulls in t_stage with median 4.0
Filled 20 nulls in n_stage with median 1.0


/var/folders/x8/3g2kg4zd76gf5ghscnz510_r0000gp/T/ipykernel_63761/672912237.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['t_stage'] = df['t_stage'].replace(t_encode)
/var/folders/x8/3g2kg4zd76gf5ghscnz510_r0000gp/T/ipykernel_63761/672912237.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['n_stage'] = df['n_stage'].replace(n_encode)
/var/folders/x8/3g2kg4zd76gf5ghscnz510_r0000gp/T/ipykernel_63761/672912237.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version

In [151]:
clinical.head()

,t_stage,n_stage,m_stage
sample,,,
TCGA-BH-A0W3-01A,3.0,2.0,0
TCGA-AR-A24V-01A,4.0,2.0,0
TCGA-E9-A1NE-01A,4.0,2.0,0
TCGA-AC-A8OQ-01A,4.0,2.0,0
TCGA-AC-A23C-01A,4.0,2.0,0


In [152]:
clinical.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1110 entries, TCGA-BH-A0W3-01A to TCGA-A7-A0D9-01A
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   t_stage  1110 non-null   float64
 1   n_stage  1110 non-null   float64
 2   m_stage  1110 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 67.0+ KB


In [153]:
# STEP 13 - align with preprocessed gene dataset patients

gene = pd.read_csv("../data/processed/gene_data.csv", index_col=0)

print("Clinical shape = ", clinical.shape)
print("Gene shape = ", gene.shape)

# find samples that we need to drop from clinical for alignment with gene
idx_not_in_gene = [x for x in clinical.index if x not in gene.index]
print(len(idx_not_in_gene))
print(idx_not_in_gene[:5])

# ensure we have no samples missing from clinical that are in gene
idx_not_in_clinical = [x for x in gene.index if x not in clinical.index]
print(len(idx_not_in_clinical))
print(idx_not_in_clinical)  # expect none

# perform inner join
merged_data = gene.join(clinical, how='inner')
final_clinical = merged_data[['t_stage', 'n_stage', 'm_stage']]

final_clinical.head()

print("Gene shape = ", gene.shape)
print("Final clinical shape = ", final_clinical.shape)

Clinical shape =  (1110, 3)
Gene shape =  (1059, 11386)
51
['TCGA-A7-A26I-01B', 'TCGA-A8-A090-01A', 'TCGA-AC-A3OD-01B', 'TCGA-A8-A081-01A', 'TCGA-A8-A083-01A']
0
[]
Gene shape =  (1059, 11386)
Final clinical shape =  (1059, 3)


In [154]:
# Save final clinical data
final_clinical.to_csv("../data/processed/clinical_data.csv")